# CAMO-X: Concealment-Aware, Material-gated, Open-set X-ray threat recognition on STCray

# Part A - Start of every session

### A.1 Environment check

In [ ]:
import os, sys, platform, importlib, time, json, warnings, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

print(platform.platform(), "| Python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | {p.total_memory / 1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no CUDA GPU visible. Install the CUDA build of PyTorch (Section 0) unless this is a smoke test.")
for pkg, pipname in [("timm", "timm"), ("cv2", "opencv-python"), ("sklearn", "scikit-learn"), ("scipy", "scipy"),
                     ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl"), ("tqdm", "tqdm")]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:12s} {getattr(m, '__version__', 'ok')}")
    except ImportError:
        print(f"{pkg:12s} MISSING -> pip install {pipname}")
for pkg in ["ultralytics", "open_clip"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:12s} {getattr(m, '__version__', 'ok')} (optional)")
    except ImportError:
        print(f"{pkg:12s} not installed (only needed for an optional baseline)")

### A.2 The `camox` library
The project code lives in a small package next to this notebook (`camox/*.py`); the cell below imports it.
Real modules are needed on Windows because DataLoader workers are started with *spawn* and cannot load
classes defined inside a notebook.

In [ ]:
import os
PKG = os.path.abspath("camox")
assert os.path.isfile(os.path.join(PKG, "__init__.py")), f"camox package not found at {PKG}"
print("library at", PKG, "|",
      len([f for f in os.listdir(PKG) if f.endswith(".py")]), "modules")


In [ ]:
import importlib
import camox
from camox import (config as C, utils as U, indexing as IX, splits as SP, material as MA, cache as CA,
                   datasets as DS, models as MO, losses as LO, evaluate as EV, engine as EN, anomaly as AN,
                   cascade as CS, baselines as BL, viz as VZ, synthetic as SY, pipeline as PL, study as ST)
for _m in (C, U, IX, SP, MA, CA, DS, MO, LO, EV, EN, AN, CS, BL, VZ, SY, PL, ST):
    importlib.reload(_m)
VZ.set_style()
print("camox", camox.__version__, "loaded")

### A.3 Settings and today's plan

In [ ]:
# ---------------------------------------------------------------- paths 
DATA_ROOT = r"dataset"      
TRAIN_DIR = r"dataset\train"                        
TEST_DIR = r"dataset\test"                         
WORK_DIR = os.path.abspath("camox_work")  

# ---------------------------------------------------------------- this session
SMOKE_TEST = False     

TODAY = ["B4_vit_s16_384", "B5_yolov8s"]

SESSION_HOURS = None  
SHOW_DATA_AUDIT = True  

# ---------------------------------------------------------------- study settings
HELDOUT = ["3D Gun", "Explosive"] 
NUM_WORKERS = 4
CACHE_SIZE = 512
VAL_FRAC = 0.15
DUP_BITS = 12
N_BOOT = 1000
BANK_SIZE = 16000
PROP_TOPK = 5                  
PROP_MAX_PER_SCAN = 3.0        
FULL = dict(epochs=20, train_fraction=1.0)   
ABL = dict(epochs=10, train_fraction=0.4)    
S2_EPOCHS = 12
YOLO_EPOCHS = 30

# ---------------------------------------------------------------- automated-test overrides
if os.environ.get("CAMOX_SMOKE") == "1":
    SMOKE_TEST = True
if os.environ.get("CAMOX_WORK"):
    WORK_DIR = os.environ["CAMOX_WORK"]
_env_today = os.environ.get("CAMOX_TODAY")
if _env_today is not None:
    TODAY = "all" if _env_today == "all" else [t for t in _env_today.split(",") if t]
if os.environ.get("CAMOX_HOURS"):
    SESSION_HOURS = float(os.environ["CAMOX_HOURS"])

if SMOKE_TEST:
    WORK_DIR = os.path.join(WORK_DIR, "smoke_test")
    DATA_ROOT, TRAIN_DIR, TEST_DIR = os.path.join(WORK_DIR, "synthetic_stcray"), None, None
    if not os.path.isfile(os.path.join(DATA_ROOT, "_done.flag")):
        shutil.rmtree(DATA_ROOT, ignore_errors=True)
        SY.make_fake_stcray(DATA_ROOT, n_train=5, n_test=3, n_multi=(10, 6), n_benign=(10, 6), seed=0)
    if _env_today is None:
        TODAY = "all"
    NUM_WORKERS, CACHE_SIZE, N_BOOT, BANK_SIZE = 0, 256, 50, 400
    FULL = dict(epochs=1, train_fraction=1.0)
    ABL = dict(epochs=1, train_fraction=0.6)
    S2_EPOCHS, YOLO_EPOCHS = 1, 1

ctx = PL.Ctx(WORK_DIR, cache_size=CACHE_SIZE, smoke=SMOKE_TEST, num_workers=NUM_WORKERS, n_boot=N_BOOT)
SETTINGS = dict(smoke=SMOKE_TEST, num_workers=NUM_WORKERS, full=FULL, abl=ABL, s2_epochs=S2_EPOCHS,
                yolo_epochs=YOLO_EPOCHS, bank_size=BANK_SIZE, heldout=HELDOUT, prop_topk=PROP_TOPK,
                prop_max=PROP_MAX_PER_SCAN)
print("SMOKE_TEST =", SMOKE_TEST, "| WORK_DIR =", ctx.work, "| device =", U.get_device())
print("TODAY =", TODAY, "| SESSION_HOURS =", SESSION_HOURS)

In [ ]:
DF, DIAG, DONORS = ST.prepare_data(ctx, DATA_ROOT, TRAIN_DIR, TEST_DIR, val_frac=VAL_FRAC, dup_bits=DUP_BITS,
                                   workers=8)

### A.5 Experiment registry and progress
`state` is one of *done*, *partial (k/N epochs)*, *trained, not evaluated* or *not started*. 

In [ ]:
study = ST.Study(ctx, SETTINGS)
study.set_session(SESSION_HOURS)
if TODAY != "all":
    unknown = [n for n in TODAY if n not in study.specs]
    if unknown:
        raise ValueError(f"Unknown experiment names in TODAY: {unknown}")
    print(f"today's plan: {TODAY} - about {sum(study.estimate(n) for n in TODAY) / 60:.1f} h still needed")
_ = study.status()

# Part B - Data audit and exploratory analysis

### B.1 Annotation audit (LabelMe)

In [ ]:
if SHOW_DATA_AUDIT:
    ST.annotation_audit(DF, DIAG)

### B.2 Class balance, clean and multi-threat shares, sizes, captions

In [ ]:
if SHOW_DATA_AUDIT:
    ST.eda(ctx, DF)

### B.3 Split and leakage audit
Near-identical re-scans of the same bag are kept on one side of the train/validation split. The official test
set is untouched; the count of test scans that have a near-duplicate in training is reported.

In [ ]:
if SHOW_DATA_AUDIT:
    ST.split_summary(ctx)

### B.4 Material prior and box placement
The hue histogram should peak near the prior's centres (orange / green / blue). In the sample grid, the green
boxes and mask outlines must sit on the threats.

In [ ]:
if SHOW_DATA_AUDIT:
    ST.material_check(ctx)

### B.5 CA-TIP preview (threat projection always on, to check the augmentation)

In [ ]:
if SHOW_DATA_AUDIT:
    ST.tip_preview(ctx, study.specs["S1_main"].cfg["img_size"])

# Part C - Experiments
One cell per experiment. A cell runs its experiment only when the name is in `TODAY`.

## C.1 Main model (Stage 1, Look)
ConvNeXt-Nano + material stream with zero-initialised gated fusion, FPN, CSRA class head and mask head, trained
with ASL + (BCE + Dice) and CA-TIP on the full training set. The checkpoint with the best validation mAP is kept.

In [ ]:
# CAMO-X Stage 1 (Look), full budget (~2.5 h on an RTX 4060)
study.cell("S1_main", TODAY)

## C.2 Baselines
B1-B4: plain fine-tuned classifiers (average pooling + BCE), same resolution and budget. B5: YOLOv8s on the same
512 px cache (bag score per class = highest box confidence). B6: zero-shot CLIP. B7 is the clean-bag anomaly bank
`D1_clean` in C.4.

In [ ]:
# B1 ResNet-50 (GAP, BCE) (~2.5 h on an RTX 4060)
study.cell("B1_resnet50", TODAY)

In [ ]:
# B2 ConvNeXt-Nano (GAP, BCE) (~2.0 h on an RTX 4060)
study.cell("B2_convnext_nano", TODAY)

In [ ]:
# B3 EfficientNet-B4 (GAP, BCE) (~3.5 h on an RTX 4060)
study.cell("B3_effnet_b4", TODAY)

In [ ]:
# B4 ViT-S/16 @384 (~2.5 h on an RTX 4060)
study.cell("B4_vit_s16_384", TODAY)

In [ ]:
# B5 YOLOv8s detector (~2.5 h on an RTX 4060) - optional, pip install ultralytics
study.cell("B5_yolov8s", TODAY)

In [ ]:
# B6 zero-shot CLIP ViT-B/16 (~6 min on an RTX 4060) - optional, pip install open_clip_torch
study.cell("B6_clip", TODAY)

## C.3 Stage 1 ablations
Every ablation changes one thing from `A0_ref` and uses the same reduced budget (`ABL`), training subset and seed.
`A0_ref` is only needed for the comparison, so run it in the same session as your first ablation (or any time
later: the comparisons are filled in as soon as it finishes).

In [ ]:
# A0 full Stage 1 (reference, reduced budget) (~32 min on an RTX 4060)
study.cell("A0_ref", TODAY)

In [ ]:
# A1 no material stream (~32 min on an RTX 4060)
study.cell("A1_no_material", TODAY)

In [ ]:
# A2 material as 9-channel input (~32 min on an RTX 4060)
study.cell("A2_early_fusion", TODAY)

In [ ]:
# A3 plain addition, no gate (~32 min on an RTX 4060)
study.cell("A3_add_fusion", TODAY)

In [ ]:
# A4 greyscale input (~32 min on an RTX 4060)
study.cell("A4_grayscale", TODAY)

In [ ]:
# A5 + hue/saturation jitter (~32 min on an RTX 4060)
study.cell("A5_hue_jitter", TODAY)

In [ ]:
# A6 no mask head (~32 min on an RTX 4060)
study.cell("A6_no_mask_head", TODAY)

In [ ]:
# A7 average pooling instead of CSRA (~32 min on an RTX 4060)
study.cell("A7_gap_head", TODAY)

In [ ]:
# A8 BCE instead of ASL (~32 min on an RTX 4060)
study.cell("A8_bce", TODAY)

In [ ]:
# A9 focal loss (~32 min on an RTX 4060)
study.cell("A9_focal", TODAY)

In [ ]:
# A10 class-balanced BCE (~32 min on an RTX 4060)
study.cell("A10_cb_bce", TODAY)

In [ ]:
# A11 no CA-TIP (~32 min on an RTX 4060)
study.cell("A11_no_tip", TODAY)

In [ ]:
# A12 TIP with random placement (~32 min on an RTX 4060)
study.cell("A12_random_tip", TODAY)

In [ ]:
# A13 384 px input (~20 min on an RTX 4060)
study.cell("A13_384px", TODAY)

### Ablation summary so far
Change in test mAP versus `A0_ref` with 95% paired bootstrap intervals, for every finished ablation. An interval
that crosses zero means the difference is not reliable.

In [ ]:
study.ablation_summary()

## C.4 Anomaly branch (open-set safety net)
PatchCore-style nearest-neighbour distances to a bank of threat-free patches from a frozen WideResNet-50. The
default clutter bank uses every patch at least two cells away from an annotated threat, from all training scans.

In [ ]:
# D1 clutter bank (default) (~12 min on an RTX 4060)
study.cell("D1_clutter", TODAY)

In [ ]:
# B7 / D1 clean-bag bank (~5 min on an RTX 4060)
study.cell("D1_clean", TODAY)

In [ ]:
# D1 clean + clutter bank (~12 min on an RTX 4060)
study.cell("D1_both", TODAY)

In [ ]:
# D2 random subsample instead of coreset (~10 min on an RTX 4060)
study.cell("D2_random", TODAY)

In [ ]:
# D2 bank of 4k patches (~10 min on an RTX 4060)
study.cell("D2_4k", TODAY)

In [ ]:
# D3 layer 3 features only (~12 min on an RTX 4060)
study.cell("D3_layer3", TODAY)

In [ ]:
study.anomaly_summary()

## C.5 Stage 2 (Zoom) and the cascade
`S2_zoom` first tunes the mask threshold for proposals on validation, cuts zoom patches from the original scans
(true threats, background, Stage 1 false alarms) and trains the crop classifier. `C1_fusion` evaluates the cascade
(Stage 1 only / Stage 2 only / average / max fusion, oracle proposals, top-K and crop-context variants, box AP).

In [ ]:
# Stage 2 (Zoom) classifier (~60 min on an RTX 4060, needs S1_main)
study.cell("S2_zoom", TODAY)

In [ ]:
# Stage 2 without the material stream (~40 min on an RTX 4060, needs S1_main)
study.cell("S2_zoom_nomat", TODAY)

In [ ]:
# C1/C3 cascade fusion, oracle and box AP (~25 min on an RTX 4060, needs S1_main + S2_zoom)
study.cell("C1_fusion", TODAY)

In [ ]:
# C2 mask + anomaly-map proposals (~15 min on an RTX 4060, needs S1_main + S2_zoom + D1_clutter)
study.cell("C2_anomaly_proposals", TODAY)

In [ ]:
# C4 Stage 2 without material (~5 min on an RTX 4060, needs S1_main + S2_zoom_nomat)
study.cell("C4_no_material", TODAY)

## C.6 Unseen-threat experiment (RQ4)
A Stage 1 model and a clutter bank are rebuilt without any scan that contains the `HELDOUT` classes. Bags whose
only threats are held-out classes are then scored by the known-class score, the anomaly score and their maximum.

In [ ]:
# E1 Stage 1 trained without 3D Gun, Explosive (~30 min on an RTX 4060)
study.cell("E1_open_s1", TODAY)

In [ ]:
# E1 clutter bank without the held-out classes (~12 min on an RTX 4060)
study.cell("E1_open_bank", TODAY)

In [ ]:
# E1 unseen-threat evaluation (~3 min on an RTX 4060, needs E1_open_s1 + E1_open_bank)
study.cell("E1_eval", TODAY)

## C.7 Optional: seed repeats (mean and spread for the main model and B2)

In [ ]:
# CAMO-X Stage 1, seed 1 (~2.5 h on an RTX 4060)
study.cell("S1_main_seed1", TODAY)

In [ ]:
# B2 ConvNeXt-Nano, seed 1 (~2.0 h on an RTX 4060)
study.cell("B2_convnext_nano_seed1", TODAY)

In [ ]:
# CAMO-X Stage 1, seed 2 (~2.5 h on an RTX 4060)
study.cell("S1_main_seed2", TODAY)

In [ ]:
# B2 ConvNeXt-Nano, seed 2 (~2.0 h on an RTX 4060)
study.cell("B2_convnext_nano_seed2", TODAY)

## C.8 Progress after this session

In [ ]:
_ = study.status()

# Part D - Results and analysis 

### D.1 Headline and full result tables

In [ ]:
_ = study.headline()

### D.2 Per-class AP, threat recall vs. clean-bag clearance, confusion among look-alikes

In [ ]:
study.per_class_and_curves()

### D.3 Difficulty strata: object size, clutter around the threat, caption clutter level, near-duplicates

In [ ]:
study.strata()

### D.4 Speed on this GPU

In [ ]:
study.latency()

### D.5 Seed spread

In [ ]:
study.seed_summary()

### D.6 Qualitative analysis
Confident hits, worst misses and worst false alarms, each with the true mask, Stage 1's mask, class maps, the
anomaly map, the proposals and Stage 2's zoom crops (whatever exists so far), plus galleries of missed threats and
of unseen threats that were caught or missed.

In [ ]:
study.qualitative()

### D.7 Export tables for the report (CSV, Markdown, LaTeX)

In [ ]:
study.export()

In [ ]:
import pandas as pd
for t in ["cascade", "proposals", "box_ap"]:
    print(t); print(pd.read_csv(f"{WORK_DIR}/results/{t}.csv").to_string(), "\n")